In [1]:
import torch
import pickle
import numpy as np
import pandas as pd
import os
import json 
from os.path import dirname



root_path = dirname(os.getcwd()) 

pd.set_option("display.max_columns", None)
# data_dir = root_path + "/data/datasets/original/"
data_dir_processed = root_path + "/data/datasets/_processed/"
data_dir_graphs = root_path + "/data/datasets/graphs_repair/"

print(root_path, data_dir_processed, data_dir_graphs, sep="\n")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
#device = "cpu"

/home/sebastiano.dissegna/NeuroSymbolic/RepairW_LTLf/NeuroSymEventLogRepair
/home/sebastiano.dissegna/NeuroSymbolic/RepairW_LTLf/NeuroSymEventLogRepair/data/datasets/_processed/
/home/sebastiano.dissegna/NeuroSymbolic/RepairW_LTLf/NeuroSymEventLogRepair/data/datasets/graphs_repair/


In [2]:
ACT_TIME_ONLY = True

In [ ]:
#ACT_TIME_ONLY = False

In [3]:
with open("../data/dataset_features.json", 'r') as file:
    datasets_info = json.load(file)

In [ ]:
list(datasets_info.keys())

In [4]:
dataset = "bpi_2013_CZ"

In [5]:
with open("../data/dataset_features.json", 'r') as file:
    dataset_info = json.load(file)[dataset]

In [6]:
tab_all = pd.read_csv(f"{data_dir_processed}{dataset}/{dataset}_processed_all.csv") 
tab_all.head()

,CaseID,Activity,org:resource,time:timestamp,concept:name,impact,lifecycle:transition,org:group,org:role,organization country,organization involved,product,resource country
0,1,Queued-Awaiting Assignment,Value 1,0.000000,Queued,High,Awaiting Assignment,Org line A2,A2_2,se,J11 2nd,PROD191,INDIA
1,1,Accepted-In Progress,Value 1,19.087576,Accepted,High,In Progress,Org line A2,A2_2,se,J11 2nd,PROD191,INDIA
2,1,Accepted-Assigned,Value 1,19.087576,Accepted,High,Assigned,Org line A2,A2_2,se,J11 2nd,PROD191,INDIA
3,1,Accepted-In Progress,Value 1,19.087580,Accepted,High,In Progress,Org line A2,A2_2,se,J11 2nd,PROD191,INDIA
4,1,Completed-Closed,Value 1,19.087581,Completed,High,Closed,Org line A2,A2_2,se,J11 2nd,PROD191,INDIA


In [7]:
print(f"number of events {len(tab_all)}")
print(f"number of traces {len(tab_all['CaseID'].unique())}")
print(f"number of attributes {len(tab_all.columns) - 1}") 

number of events 6660
number of traces 1487
number of attributes 12


In [8]:
MISSING_VALUE = "MISSING_VALUE"

In [9]:
if ACT_TIME_ONLY:
    categorical_columns = ["Activity"]
    real_value_columns = ["time:timestamp"]
    dataset = f"{dataset}_AT_only"
else:
    categorical_columns = dataset_info["categorical"]
    real_value_columns = dataset_info["numerical"]

In [10]:
list_unique = {k : list(tab_all[k].unique()) + [MISSING_VALUE] for k in categorical_columns}
#list_unique

In [11]:
import random

torch.manual_seed(0)
torch.cuda.manual_seed(0)
random.seed(0)
np.random.seed(0)

In [12]:
with open(data_dir_graphs + dataset + "_TRAIN_V2_repair.pkl", "rb") as f:
    X_train = pickle.load(f)
with open(data_dir_graphs + dataset + "_VALID_V2_repair.pkl", "rb") as f:
    X_valid = pickle.load(f)
with open(data_dir_graphs + dataset + "_TEST_V2_repair.pkl", "rb") as f:
    X_test = pickle.load(f)

In [13]:
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import ToUndirected

transform = ToUndirected()


with torch.no_grad():
        for i in range(len(X_train)):
                X_train[i] = transform(X_train[i])
        for i in range(len(X_valid)):
                X_valid[i] = transform(X_valid[i])
        for i in range(len(X_test)):
                X_test[i] = transform(X_test[i])

In [14]:
edge_types = set()
node_types = set()
for i in range(len(X_train)):
    n, edge_type = X_train[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)
for i in range(len(X_valid)):
    n, edge_type = X_valid[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)
for i in range(len(X_test)):
    n, edge_type = X_test[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)
        
node_types = list(node_types)
edge_types = list(edge_types)

## Hyperopt

In [15]:
from ax.service.managed_loop import optimize
from torch_geometric.nn import (
    HeteroConv,
    SAGEConv,
    GATv2Conv
)
from torch.nn import (
    ModuleList,
    Module,
    Linear,
    ModuleDict
  )
from typing_extensions import Self

In [16]:
import torch
import torch.nn as nn
from torch.nn import Module, ModuleList, ModuleDict, Linear
from torch_geometric.loader import DataLoader
from torch_geometric.nn import HeteroConv, SAGEConv
from tqdm.notebook import tqdm


In [17]:
# ----------------------------------------------------------------------------
# Model
# ----------------------------------------------------------------------------
class HGNN(Module):
    def __init__(self, output_cat, output_real, nodes_relations, parameters):
        super().__init__()
        hid = parameters["hid"]
        layers = parameters["layers"]
        aggregation = parameters["aggregation"]
 
        self.output_cat = output_cat
        self.output_real = output_real
 
        self.convs = ModuleList()
        for _ in range(layers):
            self.convs.append(
                HeteroConv(
                    {
                        rel: SAGEConv((-1, -1), out_channels=hid, aggr=aggregation, normalize=True)
                        #rel: GATv2Conv((-1, -1), out_channels=hid, aggr=aggregation, residual=True, add_self_loops=False)
                        
                        for rel in nodes_relations
                    },
                    aggr=aggregation,
                )
            )
 
        self.FC = ModuleDict()
        for k in output_cat:
            self.FC[k] = Linear(hid, output_cat[k])
        for k in output_real:
            self.FC[k] = Linear(hid, 1)
 
    def forward(self, batch, idx):
        """idx: dict {task: LongTensor of node indices to predict}.
        Pass precomputed indices (from masks, ~masks, or `full`)."""
        x_dict = batch.x_dict  # local copy: batch is not mutated
        for conv in self.convs:
            x_dict = {k: v.relu() for k, v in conv(x_dict, batch.edge_index_dict).items()}
 
        out = {}
        for k in self.output_cat:
            out[k] = self.FC[k](x_dict[k].index_select(0, idx[k]))
        for k in self.output_real:
            out[k] = self.FC[k](x_dict[k].index_select(0, idx[k])).squeeze(-1)
        return out

 

 


In [18]:

outputcat = {k : len(list_unique[k]) for k in list_unique}
outputreal = real_value_columns

if ACT_TIME_ONLY:
    outputcat= {"Activity" : outputcat["Activity"]}
    outputreal = ['time:timestamp']

print(outputcat)
print(outputreal)

{'Activity': 8}
['time:timestamp']


## LTLf semantic loss (variant B)
Mines DECLARE rules on the complete training traces, compiles them to DFAs and builds `sem_loss`. `LAMBDA_SEM` weights the term added to the training loss; set it to `0.0` for the baseline.

In [19]:
# Variant B (semantic loss) -- needs ltlf_features.py and ltlf_semantic_loss.py next to this notebook.
from collections import Counter
import ltlf_features as LF

LAMBDA_SEM = 0.3     # weight of the semantic loss; 0.0 = baseline (no rules used)

ACT_KEY   = "Activity"
N_CLASSES = outputcat[ACT_KEY]              # 8 here: width of the Activity one-hot = model output size

def infer_missing_id(graphs, n=2000):
    """Class index used to encode masked Activity nodes (read from the data, not assumed)."""
    c = Counter()
    for g in graphs[:n]:
        m = g.masks[ACT_KEY].bool()
        c.update(g[ACT_KEY].x[m][:, :N_CLASSES].argmax(1).tolist())
    assert len(c) == 1, f"masked Activity nodes use several encodings: {c}"
    return next(iter(c))

MISSING_ID = infer_missing_id(X_train)                      # in your example graph: 5
CLASS_IDS  = [c for c in range(N_CLASSES) if c != MISSING_ID]   # real activities, in graph-class order
A_REAL     = len(CLASS_IDS)                                  # 7
_lut = torch.full((N_CLASSES,), -1, dtype=torch.long); _lut[CLASS_IDS] = torch.arange(A_REAL)
print("MISSING class id:", MISSING_ID, "| real activities:", A_REAL)

# class index -> compact DFA symbol 0..A_REAL-1 (the missing class is NOT a symbol of any rule)
def compact(ids):                     return _lut[torch.as_tensor(ids)].tolist()

def act_ids(g):
    """Compact activity id per Activity node; masked nodes -> A_REAL (what ltlf_features calls missing_id).
    No labels used."""
    cls = g[ACT_KEY].x[:, :N_CLASSES].argmax(1)
    out = _lut[cls]
    out[cls == MISSING_ID] = A_REAL
    return out.tolist()

# --- sanity checks on training graphs ---
for g in X_train[:200]:
    m, cls, y = g.masks[ACT_KEY].bool(), g[ACT_KEY].x[:, :N_CLASSES].argmax(1), g.y[ACT_KEY]
    assert (cls[m] == MISSING_ID).all() and (cls[~m] != MISSING_ID).all(), "masked <-> MISSING class mismatch"
    assert (cls[~m] == y[~m]).all(),            "x.argmax != y on visible nodes"
    assert (y != MISSING_ID).all(),             "labels contain the MISSING class"
    # temporal order: followed_by edges must connect consecutive node indices
    ei = g[(ACT_KEY, "followed_by", ACT_KEY)].edge_index
    assert ((ei[0] - ei[1]).abs() == 1).all(),  "nodes are not stored in consecutive (temporal) order"

# --- mine rules on COMPLETE train traces (labels), in compact ids ---
# Richer patterns (alternate/succession, counts, bounded response, n-gram next-event / forbidden factors, ...).
# Rules are chosen by how many WRONG activities they eliminate at simulated masked nodes, not just by support.
import ltlf_patterns as LP
full_seqs = [compact(g.y[ACT_KEY]) for g in X_train]
print("--- original nine templates (reference) ---")
_ = LP.mine_rich_rules(full_seqs, A_REAL, families=("basic",), top_k=60)
print("--- richer patterns ---")
rules = LP.mine_rich_rules(full_seqs, A_REAL, families=LP.FAMILIES, top_k=60)   # [(label, dfa, support)]
dfas  = [d for _, d, _ in rules]
names = sorted(list_unique[ACT_KEY])
if names[MISSING_ID] == "MISSING_VALUE":
    names = [names[c] for c in CLASS_IDS]; print("symbol -> activity:", dict(enumerate(names)))
for lab, _, s in rules[:15]: print(round(s, 3), lab)

import ltlf_semantic_loss as SL
sem_loss = SL.LTLfSemanticLoss(dfas, CLASS_IDS, N_CLASSES, act_key=ACT_KEY,
                               rule_weights=[s for _, _, s in rules]).to(device)


MISSING class id: 5 | real activities: 7
--- original nine templates (reference) ---
273 candidate patterns, 103 with support >= 0.98
selected 21 rules | eliminate 60.9% of wrong (node, activity) pairs, true activity eliminated at 2.31% of masked nodes (simulation: 2852 masked nodes)
--- richer patterns ---
844 candidate patterns, 311 with support >= 0.98
selected 49 rules | eliminate 67.5% of wrong (node, activity) pairs, true activity eliminated at 2.31% of masked nodes (simulation: 2852 masked nodes)
symbol -> activity: {0: 'Accepted-Assigned', 1: 'Accepted-In Progress', 2: 'Accepted-Wait', 3: 'Completed-Cancelled', 4: 'Completed-Closed', 5: 'Queued-Awaiting Assignment', 6: 'Unmatched-Unmatched'}
0.983 chain_succession(6,3)
0.999 succession(1,4)
0.983 chain_succession(3,6)
0.993 bounded_response(5,1,5)
0.983 precedence(1,2)
0.986 not_succession(4,0)
0.999 alternate_precedence(1,4)
1.0 end(4)
1.0 not_chain_succession(5,2)
0.99 not_succession(4,2)
1.0 factor_next_in((0, 0),[0, 1, 2, 5

In [20]:
# ===== CELL C (logic layer: conditioning on the active rules) — after CELL 0 =====
import ltlf_logic_layer as LL
LAMBDA_COND = 1.0          # weight of the conditioned-output CE (raw CE stays under GradNorm); 0 = layer off in training
LAYER_SUP_MIN = 0.99       # rules used by the layer / hard decoder: train support >= this
cnt_ = Counter(tuple(s) for s in full_seqs)
train_sup = np.array([sum(c for s, c in cnt_.items() if d.accepts(s)) / len(full_seqs) for d in dfas])
layer_idx = [k for k, s in enumerate(train_sup) if s >= LAYER_SUP_MIN]
logic_layer = LL.LTLfLogicLayer([dfas[k] for k in layer_idx], CLASS_IDS, N_CLASSES,
                                act_key=ACT_KEY, n_iter=1).to(device)
print(f"logic layer: {len(layer_idx)} rules, Q={logic_layer.bank.Q}")
# then the train_hgnn edits and the evaluation in test_multi_layer.py (config["lambda_cond"] = LAMBDA_COND)


logic layer: 39 rules, Q=7


In [21]:
import torch
import torch.nn as nn
from torch.nn.functional import l1_loss
from torch_geometric.loader import DataLoader
from torcheval.metrics.functional import multiclass_accuracy
 
# Expected globals from your notebook: X_test, device
 
 
def _indices(x, keys, mode):
    """Node indices to evaluate per task. Runs on the CPU batch (no GPU sync)."""
    if mode == "full":      # every node of that type
        return {k: torch.arange(x.y[k].shape[0]) for k in keys}
    if mode == "correct":   # nodes NOT masked
        return {k: (~x.masks[k]).nonzero().squeeze(1) for k in keys}
    if mode == "missing":   # masked nodes (same as training/validation)
        return {k: x.masks[k].nonzero().squeeze(1) for k in keys}
    raise ValueError(f"unknown mode {mode!r}")
 
 
@torch.no_grad()
def evaluate_hgnn(net, output_cat, output_real, test_graphs=None,
                  mode="missing", batch_size=128, num_workers=4):
    if test_graphs is None:
        test_graphs = X_test
 
    keys = list(output_cat) + list(output_real)
    criteria = {k: nn.CrossEntropyLoss() for k in output_cat}
    criteria.update({k: nn.L1Loss() for k in output_real})
 
    loader = DataLoader(
        test_graphs, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
        persistent_workers=num_workers > 0,
    )
 
    pred_cat = {k: [] for k in output_cat}
    tgt_cat = {k: [] for k in output_cat}
    pred_real = {k: [] for k in output_real}
    tgt_real = {k: [] for k in output_real}
    batch_losses = []  # one [n_tasks] tensor per batch, kept on GPU
 
    net.eval()
    for x in loader:
        idx = _indices(x, keys, mode)                     # CPU
        x = x.to(device, non_blocking=True)
        idx = {k: v.to(device, non_blocking=True) for k, v in idx.items()}
 
        labels = {k: x.y[k].index_select(0, idx[k]) for k in keys}
        out = net(x, idx)
 
        batch_losses.append(torch.stack([criteria[k](out[k], labels[k]) for k in keys]))
 
        for k in output_cat:
            pred_cat[k].append(out[k].argmax(dim=1))      # softmax is monotonic: not needed
            tgt_cat[k].append(labels[k])
        for k in output_real:
            pred_real[k].append(out[k])
            tgt_real[k].append(labels[k])
 
    # single GPU -> CPU sync for all per-batch losses
    L = torch.stack(batch_losses).cpu()                   # [n_batches, n_tasks]
    per_task = dict(zip(keys, L.mean(0).tolist()))        # mean of per-batch losses
    avg_total = L.sum(1).mean().item()
 
    accuracy = {
        k: multiclass_accuracy(
            torch.cat(pred_cat[k]), torch.cat(tgt_cat[k]), num_classes=output_cat[k]
        ).item()
        for k in output_cat
    }
    mae_global = {
        k: l1_loss(torch.cat(pred_real[k]), torch.cat(tgt_real[k])).item()
        for k in output_real
    }
 
    res = (
        {f"{k}_acc": accuracy[k] for k in output_cat}
        | {f"{k}_mae": per_task[k] for k in output_real}
        | {"AVG_total_loss": avg_total}
        | {f"MAE_{k}": mae_global[k] for k in output_real}
    )
    print(res)
    return res
 

# Drop-in replacements for your three functions
def test_hgnn_full(net, output_cat, output_real, test_graphs=None, **kw):
    return evaluate_hgnn(net, output_cat, output_real, test_graphs, mode="full", **kw)
 
 
def test_correct_hgnn(net, output_cat, output_real, test_graphs=None, **kw):
    return evaluate_hgnn(net, output_cat, output_real, test_graphs, mode="correct", **kw)
 
 
def test_misssing_hgnn(net, output_cat, output_real, test_graphs=None, **kw):
    return evaluate_hgnn(net, output_cat, output_real, test_graphs, mode="missing", **kw)

In [22]:
from torcheval.metrics.functional import multiclass_accuracy, multiclass_f1_score
import torch.nn as nn
from copy import deepcopy
from tqdm.notebook import tqdm

# ----------------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------------
def mask_to_idx(batch, keys):
    """Run on CPU batch BEFORE .to(device): nonzero() causes no GPU sync there."""
    return {k: batch.masks[k].nonzero().squeeze(1) for k in keys}
 
 
def grad_norm(loss, params):
    g = torch.autograd.grad(loss, params, retain_graph=True, allow_unused=True)
    sq = [t.pow(2).sum() for t in g if t is not None]
    return torch.stack(sq).sum().sqrt() if sq else loss.new_zeros(())
 
 
# ----------------------------------------------------------------------------
# Training
# ----------------------------------------------------------------------------
def train_hgnn(config, output_cat, output_real, epochs=20, callback=None):
    print(config)
 
    use_amp = config.get("amp", True)                 # bf16 autocast
    gn_every = config.get("gn_every", 1)              # GradNorm update every k steps
    num_workers = config.get("num_workers", 4)
    lam = config.get("lambda_sem", 0.0)               # weight of the LTLf semantic loss (0 = baseline)
    lam_c = config.get("lambda_cond", 0.0)            # weight of CE on the logic-layer output (0 = layer off in training)
    amp_kwargs = dict(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp)
 
    net = HGNN(
        parameters=config,
        output_cat=output_cat,
        output_real=output_real,
        nodes_relations=edge_types,
    ).to(device)
 
    task_keys = list(output_cat) + list(output_real)
    n_tasks = len(task_keys)
    criteria = {k: nn.CrossEntropyLoss() for k in output_cat}
    criteria.update({k: nn.L1Loss() for k in output_real})
 
    loader_kw = dict(
        batch_size=config["batch_size"],
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=num_workers > 0,
    )
    train_loader = DataLoader(X_train, shuffle=True, **loader_kw)
    valid_loader = DataLoader(X_valid, shuffle=False, **loader_kw)
 
    def prepare(x):
        idx = mask_to_idx(x, task_keys)                       # CPU
        x = x.to(device, non_blocking=True)
        idx = {k: v.to(device, non_blocking=True) for k, v in idx.items()}
        labels = {k: x.y[k].index_select(0, idx[k]) for k in task_keys}
        return x, idx, labels
 
    def task_losses(x, idx, labels, return_out=False):
        with torch.autocast(**amp_kwargs):
            out = net(x, idx)
        l = torch.stack([criteria[k](out[k].float(), labels[k]) for k in task_keys])
        return (l, out) if return_out else l
 
    # Dry run so lazy (-1,-1) layers are initialised before optimizer / param lists
    with torch.no_grad():
        x0, i0, _ = prepare(next(iter(train_loader)))
        net(x0, i0)
        del x0, i0
 
    optimizer = torch.optim.Adam(net.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    shared = list(net.convs[-1].parameters())  # last shared layer for GradNorm
 
    # GradNorm state: created once, never rebuilt
    weights = torch.ones(n_tasks, device=device, requires_grad=True)
    w_opt = torch.optim.Adam([weights], lr=config["lr_gradNorm"])
    L0 = None
    step = 0
 
    best_state, best_loss, pat_count, patience = None, float("inf"), 0, 10
 
    for epoch in tqdm(range(epochs)):
        net.train()
        for x in train_loader:
            x, idx, labels = prepare(x)
            losses_values, out = task_losses(x, idx, labels, return_out=True)
 
            if L0 is None:
                L0 = losses_values.detach()
 
            do_gn = step % gn_every == 0
            if do_gn:
                # per-task gradient norms on last shared layer (no double backward)
                norms = torch.stack([grad_norm(losses_values[i], shared) for i in range(n_tasks)])
                gw = weights * norms                      # differentiable wrt weights only
                loss_ratio = losses_values.detach() / L0
                rt = loss_ratio / loss_ratio.mean()
                target = (gw.mean() * rt ** config["alpha"]).detach()
                gn_loss = (gw - target).abs().sum()
 
                w_opt.zero_grad(set_to_none=True)
                gn_loss.backward()                        # touches only `weights`
 
            optimizer.zero_grad(set_to_none=True)
            l_sem = sem_loss(x, idx, out["Activity"]) if lam > 0 else 0.0   # semantic loss, outside GradNorm
            l_cond = logic_layer(x, idx, out["Activity"]) if lam_c > 0 else 0.0   # CE of the conditioned output
            (weights.detach() @ losses_values + lam * l_sem + lam_c * l_cond).backward()   # frees the graph
            optimizer.step()
 
            if do_gn:
                w_opt.step()
                with torch.no_grad():
                    weights.mul_(n_tasks / weights.sum()) # renormalise in place
            step += 1
 
        # ---- validation (single sync per epoch) ----
        net.eval()
        total = torch.zeros((), device=device)
        n_batches = 0
        with torch.no_grad():
            for x in valid_loader:
                x, idx, labels = prepare(x)
                l_v, out_v = task_losses(x, idx, labels, return_out=True)
                total += l_v.sum() + (lam_c * logic_layer(x, idx, out_v["Activity"]) if lam_c > 0 else 0.0)
                n_batches += 1
        val_loss = (total / n_batches).item()
 
        print(f"Epoch {epoch}: val loss [{val_loss:.5f}] best loss [{best_loss:.5f}]")
 
        if callback is not None:
            callback(epoch, val_loss)   # may raise optuna.TrialPruned
 
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
            pat_count = 0
        else:
            pat_count += 1
            if pat_count >= patience:
                break
 
    net.load_state_dict(best_state)
    net.best_val_loss = best_loss
    return net

In [23]:
@torch.no_grad()
def satisfaction_rate(net, graphs, outputcat, outputreal, batch_size=128):
    """Share of (trace, rule) pairs satisfied when argmax predictions fill the gaps.
    Only pairs whose rule holds on the ground-truth trace are counted, so it is comparable across models."""
    keys = list(outputcat) + list(outputreal)
    net.eval(); sat = tot = 0
    for x in DataLoader(graphs, batch_size=batch_size, shuffle=False):
        idx = mask_to_idx(x, keys)
        x = x.to(device); idx = {k: v.to(device) for k, v in idx.items()}
        out = net(x, idx)
        s, t = sem_loss.hard_satisfaction(x, idx, out[ACT_KEY])
        sat += s; tot += t
    return sat / max(tot, 1)


In [24]:
import gc
import json
import math
 
import optuna
import torch
 
# Expected in the notebook: train_hgnn, evaluate_hgnn / test_* functions,
# X_train, X_valid, X_test, edge_types, device, output_cat, output_real
 
 
# ----------------------------------------------------------------------------
# Search space
# ----------------------------------------------------------------------------
def sample_config(trial):
    return {
        "hid": trial.suggest_categorical("hid", [32, 64, 128, 256]),
        "layers": trial.suggest_int("layers", 1, 4),
        "aggregation": trial.suggest_categorical("aggregation", ["mean", "sum", "max"]),
        "lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128, 256]),
        #"batch_size": trial.suggest_categorical("batch_size", [256, 512,1024]),
        # GradNorm
        "lr_gradNorm": trial.suggest_float("lr_gradNorm", 1e-3, 1e-1, log=True),
        "alpha": trial.suggest_float("alpha", 0.0, 2.0),
        #"gn_every": trial.suggest_categorical("gn_every", [1, 5, 10]),
        "gn_every": trial.suggest_categorical("gn_every", [1,2]),
        # fixed
        "amp": True,
        "num_workers": 4,
        "lambda_sem": LAMBDA_SEM,      # fixed (not tuned): set LAMBDA_SEM=0.0 for the baseline run
    }
 
 
# ----------------------------------------------------------------------------
# Objective: train on X_train, early-stop + score on X_valid (inside train_hgnn)
# ----------------------------------------------------------------------------
def make_objective(output_cat, output_real, epochs):
    def objective(trial):
        config = sample_config(trial)
        torch.manual_seed(trial.number)
 
        def callback(epoch, val_loss):
            if not math.isfinite(val_loss):
                raise optuna.TrialPruned()
            trial.report(val_loss, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()
 
        net = None
        try:
            net = train_hgnn(config, output_cat, output_real, epochs=epochs, callback=callback)
            return net.best_val_loss
        except torch.cuda.OutOfMemoryError:
            raise optuna.TrialPruned()      # config too big for the GPU
        finally:
            del net
            gc.collect()
            torch.cuda.empty_cache()
 
    return objective
 
 
def run_hpo(output_cat, output_real, n_trials=50, epochs=30, timeout=None,
            study_name="hgnn", storage=None, seed=0):
    study = optuna.create_study(
        study_name=study_name,
        storage=storage,
        load_if_exists=True,                # re-run the cell to resume
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=seed),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3),
    )
    study.optimize(
        make_objective(output_cat, output_real, epochs),
        n_trials=n_trials,
        timeout=timeout,
        gc_after_trial=True,
    )
 
    print("Best val loss:", study.best_value)
    print("Best params:", json.dumps(study.best_params, indent=2))
    try:
        print("Importances:", optuna.importance.get_param_importances(study))
    except Exception:
        pass
 
    #with open("best_params.json", "w") as f:
    #    json.dump(study.best_params, f, indent=2)
    return study


In [26]:
study = run_hpo(outputcat, outputreal, n_trials=20, epochs=300)
# Retrain best config (more epochs), still validated on X_valid for early stopping
best_config = {**study.best_params, "amp": True, "num_workers": 4, "lambda_sem": LAMBDA_SEM}
net = train_hgnn(best_config, outputcat, outputreal, epochs=300)
#torch.save(net.state_dict(), "best_hgnn.pt")
# Touch the test set only now, once
print("Test on missing")
test_misssing_hgnn(net, outputcat, outputreal)

print("Test on correct")
test_correct_hgnn(net, outputcat, outputreal)

print("Test on full")
test_hgnn_full(net, outputcat, outputreal)


[I 2026-10-28 15:27:41,967] A new study created in memory with name: hgnn


{'hid': 64, 'layers': 2, 'aggregation': 'max', 'lr': 0.008459126528049376, 'weight_decay': 1.4135935551752307e-05, 'batch_size': 256, 'lr_gradNorm': 0.0013869861245357324, 'alpha': 0.17425859940308142, 'gn_every': 2, 'amp': True, 'num_workers': 4, 'lambda_sem': 0.3}


  0%|          | 0/300 [00:00<?, ?it/s]

Epoch 0: val loss [1.51645] best loss [inf]
Epoch 1: val loss [1.31673] best loss [1.51645]
Epoch 2: val loss [1.18922] best loss [1.31673]
Epoch 3: val loss [1.10557] best loss [1.18922]
Epoch 4: val loss [1.07940] best loss [1.10557]
Epoch 5: val loss [1.10012] best loss [1.07940]
Epoch 6: val loss [1.09516] best loss [1.07940]
Epoch 7: val loss [1.04508] best loss [1.07940]
Epoch 8: val loss [1.10588] best loss [1.04508]
Epoch 9: val loss [1.02619] best loss [1.04508]
Epoch 10: val loss [1.06333] best loss [1.02619]
Epoch 11: val loss [1.01829] best loss [1.02619]
Epoch 12: val loss [1.03356] best loss [1.01829]
Epoch 13: val loss [1.04274] best loss [1.01829]
Epoch 14: val loss [1.09966] best loss [1.01829]
Epoch 15: val loss [0.99238] best loss [1.01829]
Epoch 16: val loss [1.04195] best loss [0.99238]
Epoch 17: val loss [1.04056] best loss [0.99238]
Epoch 18: val loss [1.00677] best loss [0.99238]
Epoch 19: val loss [1.12247] best loss [0.99238]
Epoch 20: val loss [1.00832] best 

[I 2026-10-28 15:27:48,375] Trial 0 finished with value: 0.992382287979126 and parameters: {'hid': 64, 'layers': 2, 'aggregation': 'max', 'lr': 0.008459126528049376, 'weight_decay': 1.4135935551752307e-05, 'batch_size': 256, 'lr_gradNorm': 0.0013869861245357324, 'alpha': 0.17425859940308142, 'gn_every': 2}. Best is trial 0 with value: 0.992382287979126.


...

[I 2026-10-28 15:38:32,833] Trial 19 finished with value: 0.8729978203773499 and parameters: {'hid': 64, 'layers': 4, 'aggregation': 'sum', 'lr': 0.0021807722296507278, 'weight_decay': 9.483868161128669e-06, 'batch_size': 32, 'lr_gradNorm': 0.010324873853098172, 'alpha': 1.8142789247528925, 'gn_every': 2}. Best is trial 12 with value: 0.8660741448402405.


Best val loss: 0.8660741448402405
Best params: {
  "hid": 64,
  "layers": 4,
  "aggregation": "sum",
  "lr": 0.007486089818615434,
  "weight_decay": 1.2152545968945812e-05,
  "batch_size": 32,
  "lr_gradNorm": 0.052443206822988225,
  "alpha": 0.8015209974999538,
  "gn_every": 2
}
Importances: {'layers': 0.5844863982215378, 'weight_decay': 0.21106733682799284, 'lr_gradNorm': 0.13242714322000143, 'alpha': 0.029769206596432216, 'batch_size': 0.015324067317321005, 'hid': 0.012354957886743761, 'aggregation': 0.01060226443121317, 'lr': 0.003960544535554462, 'gn_every': 8.080963203214563e-06}
{'hid': 64, 'layers': 4, 'aggregation': 'sum', 'lr': 0.007486089818615434, 'weight_decay': 1.2152545968945812e-05, 'batch_size': 32, 'lr_gradNorm': 0.052443206822988225, 'alpha': 0.8015209974999538, 'gn_every': 2, 'amp': True, 'num_workers': 4, 'lambda_sem': 0.3}


  0%|          | 0/300 [00:00<?, ?it/s]

Epoch 0: val loss [1.00642] best loss [inf]
Epoch 1: val loss [0.99925] best loss [1.00642]
Epoch 2: val loss [0.97960] best loss [0.99925]
Epoch 3: val loss [0.93365] best loss [0.97960]
Epoch 4: val loss [0.97586] best loss [0.93365]
Epoch 5: val loss [0.99871] best loss [0.93365]
Epoch 6: val loss [0.99515] best loss [0.93365]
Epoch 7: val loss [0.88757] best loss [0.93365]
Epoch 8: val loss [0.94001] best loss [0.88757]
Epoch 9: val loss [0.92948] best loss [0.88757]
Epoch 10: val loss [0.88855] best loss [0.88757]
Epoch 11: val loss [0.92013] best loss [0.88757]
Epoch 12: val loss [0.92376] best loss [0.88757]
Epoch 13: val loss [0.95295] best loss [0.88757]
Epoch 14: val loss [0.92093] best loss [0.88757]
Epoch 15: val loss [0.91007] best loss [0.88757]
Epoch 16: val loss [0.91810] best loss [0.88757]
Epoch 17: val loss [0.93873] best loss [0.88757]
Test on missing
{'Activity_acc': 0.6745591759681702, 'time:timestamp_mae': 0.008102229796350002, 'AVG_total_loss': 0.851981103420257

{'Activity_acc': 0.6787798404693604,
 'time:timestamp_mae': 0.007601225282996893,
 'AVG_total_loss': 0.8084844946861267,
 'MAE_time:timestamp': 0.007454421371221542}

## Test on different test sets

In [27]:
test_types = ["even", "odd", "window", "random", "attr_level"]

In [28]:
import pandas


def create_df(results):
    res = {}
    
    for k in results[0]:
        res[k] = [x[k] for x in results]
    
    res = pandas.DataFrame(data=res)
    
    return res, res.mean(), res.std()
        
    

In [30]:
import ltlf_decode as LD
import ltlf_logic_layer as LL
from collections import Counter


def test_multi(config, outputcat, outputreal, num_runs=10, num_epochs=120, sup_min=0.98, beam=16, tag=None):
    """Train num_runs models (config["lambda_cond"] > 0 trains with the logic layer) and, on every test type, compare
         raw        argmax of the network output
         cond       argmax after the logic layer (conditioning on the active rules)
         hard       hard-constrained beam decoding on the raw output
         cond_hard  hard-constrained beam decoding on the conditioned output
    The layer and the decoder use the rules with train support >= sup_min (also used in training).
    tag: name of this configuration in the files (default: 'cond{lambda_cond}')."""
    global logic_layer
    tag = tag or f"cond{config.get('lambda_cond', 0.0)}"
    save_path = f"results/results_Caise_GRADNORM/{dataset}/"
    model_path = f"trained_model/{dataset}/{tag}/"
    os.makedirs(save_path, exist_ok=True); os.makedirs(model_path, exist_ok=True)
    keys = list(outputcat) + list(outputreal)

    cnt = Counter(tuple(s) for s in full_seqs)
    train_sup = np.array([sum(c for s, c in cnt.items() if d.accepts(s)) / len(full_seqs) for d in dfas])
    ridx = [k for k, s in enumerate(train_sup) if s >= sup_min]
    sub = [dfas[k] for k in ridx]
    logic_layer = LL.LTLfLogicLayer(sub, CLASS_IDS, N_CLASSES, act_key=ACT_KEY, n_iter=config.get("layer_iters", 1)).to(device)
    decoder = LD.HardDecoder(dfas, A_REAL, rule_idx=ridx)
    print(f"{tag}: sup_min={sup_min}, {len(ridx)} rules")

    tests = {}
    for test_type in test_types:
        with open(data_dir_graphs + dataset + f"_TEST_V2_repair_{test_type}.pkl", "rb") as f:
            X = pickle.load(f)
        for j in range(len(X)):
            X[j] = transform(X[j])
        tests[test_type] = X

    res = {}
    for run in range(num_runs):
        print(f"Run {run}")
        net = train_hgnn(config, outputcat, outputreal, num_epochs)
        torch.save(net.state_dict(), f"{model_path}run{run}.pt")
        with open(f"{model_path}run{run}_config.json", "w") as f:
            json.dump({**config, "best_val_loss": float(net.best_val_loss), "sup_min": sup_min}, f, indent=2)
        for test_type, X in tests.items():
            print(f"Test type {test_type}")
            r = evaluate_hgnn(net, outputcat, outputreal, test_graphs=X, mode="missing")
            h = LL.evaluate_layer(net, X, logic_layer, decoder, CLASS_IDS, MISSING_ID, N_CLASSES, keys, device,
                                  mask_to_idx, DataLoader, beam=beam)
            if abs(h["acc_raw"] - r.get("Activity_acc", h["acc_raw"])) > 1e-3:
                print(f"  warning: raw accuracy differs ({h['acc_raw']:.4f} vs {r['Activity_acc']:.4f})")
            r.update({f"layer_{k}": v for k, v in h.items()})
            res.setdefault(test_type, []).append(r)

    summary = []
    for test_type in test_types:
        tab, means, stds = create_df(res[test_type])
        tab.to_csv(f"{save_path}{test_type}_V2_RESULTS_END_{tag}.csv", index=False)
        pd.DataFrame({"mean": means, "std": stds}).to_csv(f"{save_path}{test_type}_V2_MEAN_STD_END_{tag}.csv")
        row = {"tag": tag, "test_type": test_type, "n_missing": tab["layer_n_missing"].iloc[0],
               "active_rules": tab["layer_active_rules"].mean()}
        for n in ("raw", "cond", "hard", "cond_hard"):
            row[f"acc_{n}"] = tab[f"layer_acc_{n}"].mean(); row[f"acc_{n}_std"] = tab[f"layer_acc_{n}"].std()
            row[f"sat_{n}"] = tab[f"layer_sat_{n}"].mean()
        summary.append(row)
    summary = pd.DataFrame(summary)
    summary.to_csv(f"{save_path}{dataset}_summary_{tag}.csv", index=False)
    print(summary.round(4).to_string(index=False))
    return res


In [ ]:
dataset

In [ ]:
best_config

In [31]:
# Two configurations, same seeds/settings: plain training vs training with the logic layer (CE on raw + conditioned output).
# Both are evaluated with the layer at inference (cond), so "cond" of the baseline = inference-only layer.
res_base  = test_multi({**best_config, "lambda_cond": 0.0},         outputcat, outputreal, 10, 120, sup_min=0.98, tag="cond0.0")
res_layer = test_multi({**best_config, "lambda_cond": LAMBDA_COND}, outputcat, outputreal, 10, 120, sup_min=0.98, tag=f"cond{LAMBDA_COND}")


cond0.0: sup_min=0.98, 49 rules
Run 0
{'hid': 64, 'layers': 4, 'aggregation': 'sum', 'lr': 0.007486089818615434, 'weight_decay': 1.2152545968945812e-05, 'batch_size': 32, 'lr_gradNorm': 0.052443206822988225, 'alpha': 0.8015209974999538, 'gn_every': 2, 'amp': True, 'num_workers': 4, 'lambda_sem': 0.3, 'lambda_cond': 0.0}


  0%|          | 0/120 [00:00<?, ?it/s]

Epoch 0: val loss [1.03060] best loss [inf]
Epoch 1: val loss [0.97169] best loss [1.03060]
Epoch 2: val loss [0.99685] best loss [0.97169]
Epoch 3: val loss [0.94813] best loss [0.97169]
Epoch 4: val loss [0.93979] best loss [0.94813]
Epoch 5: val loss [0.94022] best loss [0.93979]
Epoch 6: val loss [1.04613] best loss [0.93979]
Epoch 7: val loss [0.88104] best loss [0.93979]
Epoch 8: val loss [0.92195] best loss [0.88104]
Epoch 9: val loss [0.88377] best loss [0.88104]
Epoch 10: val loss [0.90764] best loss [0.88104]
Epoch 11: val loss [0.91749] best loss [0.88104]
Epoch 12: val loss [0.93405] best loss [0.88104]
Epoch 13: val loss [0.93184] best loss [0.88104]
Epoch 14: val loss [0.91424] best loss [0.88104]
Epoch 15: val loss [0.89055] best loss [0.88104]
Epoch 16: val loss [0.89441] best loss [0.88104]
Epoch 17: val loss [0.89340] best loss [0.88104]
Test type even
{'Activity_acc': 0.7379807829856873, 'time:timestamp_mae': 0.006037659011781216, 'AVG_total_loss': 0.5699664354324341

  0%|          | 0/120 [00:00<?, ?it/s]

Epoch 0: val loss [1.01993] best loss [inf]
Epoch 1: val loss [0.97933] best loss [1.01993]
Epoch 2: val loss [0.98745] best loss [0.97933]
Epoch 3: val loss [0.95779] best loss [0.97933]
Epoch 4: val loss [0.92718] best loss [0.95779]
Epoch 5: val loss [0.91677] best loss [0.92718]
Epoch 6: val loss [0.97216] best loss [0.91677]
Epoch 7: val loss [0.91697] best loss [0.91677]
Epoch 8: val loss [0.92079] best loss [0.91677]
Epoch 9: val loss [0.91992] best loss [0.91677]
Epoch 10: val loss [0.90764] best loss [0.91677]
Epoch 11: val loss [0.99836] best loss [0.90764]
Epoch 12: val loss [0.89644] best loss [0.90764]
Epoch 13: val loss [0.88007] best loss [0.89644]
Epoch 14: val loss [0.87639] best loss [0.88007]
Epoch 15: val loss [0.87135] best loss [0.87639]
Epoch 16: val loss [0.92242] best loss [0.87135]
Epoch 17: val loss [0.89805] best loss [0.87135]
Epoch 18: val loss [0.89113] best loss [0.87135]
Epoch 19: val loss [0.89500] best loss [0.87135]
Epoch 20: val loss [0.92050] best 

...

Epoch 0: val loss [2.17818] best loss [inf]
Epoch 1: val loss [2.12904] best loss [2.17818]
Epoch 2: val loss [2.09288] best loss [2.12904]
Epoch 3: val loss [2.07537] best loss [2.09288]
Epoch 4: val loss [2.00906] best loss [2.07537]
Epoch 5: val loss [2.05892] best loss [2.00906]
Epoch 6: val loss [1.98190] best loss [2.00906]
Epoch 7: val loss [2.08871] best loss [1.98190]
Epoch 8: val loss [2.00123] best loss [1.98190]
Epoch 9: val loss [2.04000] best loss [1.98190]
Epoch 10: val loss [2.04638] best loss [1.98190]
Epoch 11: val loss [2.04050] best loss [1.98190]
Epoch 12: val loss [2.05173] best loss [1.98190]
Epoch 13: val loss [1.99049] best loss [1.98190]
Epoch 14: val loss [2.03759] best loss [1.98190]
Epoch 15: val loss [1.99471] best loss [1.98190]
Epoch 16: val loss [2.03424] best loss [1.98190]
Test type even
{'Activity_acc': 0.6899038553237915, 'time:timestamp_mae': 0.004456758964806795, 'AVG_total_loss': 0.699581503868103, 'MAE_time:timestamp': 0.004941285587847233}
Test

  0%|          | 0/120 [00:00<?, ?it/s]

Epoch 0: val loss [2.22457] best loss [inf]
Epoch 1: val loss [2.14848] best loss [2.22457]
Epoch 2: val loss [2.13321] best loss [2.14848]
Epoch 3: val loss [2.10308] best loss [2.13321]
Epoch 4: val loss [2.08627] best loss [2.10308]
Epoch 5: val loss [2.11360] best loss [2.08627]
Epoch 6: val loss [1.98843] best loss [2.08627]
Epoch 7: val loss [2.07062] best loss [1.98843]
Epoch 8: val loss [2.08154] best loss [1.98843]
Epoch 9: val loss [2.05276] best loss [1.98843]
Epoch 10: val loss [2.02186] best loss [1.98843]
Epoch 11: val loss [2.01384] best loss [1.98843]
Epoch 12: val loss [2.00607] best loss [1.98843]
Epoch 13: val loss [2.02740] best loss [1.98843]
Epoch 14: val loss [2.02347] best loss [1.98843]
Epoch 15: val loss [2.00524] best loss [1.98843]
Epoch 16: val loss [2.00024] best loss [1.98843]
Test type even
{'Activity_acc': 0.7091346383094788, 'time:timestamp_mae': 0.0055704861879348755, 'AVG_total_loss': 0.6636917591094971, 'MAE_time:timestamp': 0.005582588259130716}
Te

  0%|          | 0/120 [00:00<?, ?it/s]

Epoch 0: val loss [2.15245] best loss [inf]
Epoch 1: val loss [2.14872] best loss [2.15245]
Epoch 2: val loss [2.11814] best loss [2.14872]
Epoch 3: val loss [2.06805] best loss [2.11814]
Epoch 4: val loss [2.03447] best loss [2.06805]
Epoch 5: val loss [1.99938] best loss [2.03447]
Epoch 6: val loss [2.03547] best loss [1.99938]
Epoch 7: val loss [1.98019] best loss [1.99938]
Epoch 8: val loss [1.98627] best loss [1.98019]
Epoch 9: val loss [1.97368] best loss [1.98019]
Epoch 10: val loss [1.91911] best loss [1.97368]
Epoch 11: val loss [2.00783] best loss [1.91911]
Epoch 12: val loss [1.99764] best loss [1.91911]
Epoch 13: val loss [2.05461] best loss [1.91911]
Epoch 14: val loss [2.01015] best loss [1.91911]
Epoch 15: val loss [1.96219] best loss [1.91911]
Epoch 16: val loss [1.96421] best loss [1.91911]
Epoch 17: val loss [2.02637] best loss [1.91911]
Epoch 18: val loss [1.98246] best loss [1.91911]
Epoch 19: val loss [2.02841] best loss [1.91911]
Epoch 20: val loss [2.01646] best 

## Hard constraints at inference
`diagnose` shows how often the rules can exclude anything at all. `LD.HardDecoder` then decodes all missing activities of a trace jointly, keeping only assignments accepted by every enforced rule (beam search, exact for large beams). Rules the visible events already violate are ignored for that trace. Needs the trained `net` from the HPO/retrain cell.

In [32]:
# Do the rules carry information on this log? Per-position check on the test graphs.
# mask[i] = activities allowed at missing node i by ALL rules (a rule already violated by visible events is ignored).
def diagnose(graphs, n=3000):
    A = A_REAL
    n_miss = n_excl_any = n_excl_true = 0; allowed = []
    for g in graphs[:n]:
        acts = act_ids(g); m = [i for i, a in enumerate(acts) if a == A]
        if not m: continue
        f = LF.rule_features(dfas, acts, A).reshape(len(acts), len(dfas), A + 1)
        feas = (f[:, :, :A] > 0) | (f[:, :, A:] > 0)          # unsatisfiable rule -> no restriction
        mask = feas.all(1); y = compact(g.y[ACT_KEY])
        for i in m:
            n_miss += 1; allowed.append(int(mask[i].sum()))
            n_excl_any  += mask[i].sum() < A
            n_excl_true += not mask[i, y[i]]
    print(f"missing nodes: {n_miss} | mean #activities left (of {A}): {np.mean(allowed):.2f}")
    print(f"rules exclude at least one activity at {100*n_excl_any/n_miss:.1f}% of missing nodes")
    print(f"rules exclude the TRUE activity at   {100*n_excl_true/n_miss:.2f}% of missing nodes")
diagnose(X_test)


missing nodes: 1985 | mean #activities left (of 7): 3.13
rules exclude at least one activity at 98.6% of missing nodes
rules exclude the TRUE activity at   0.65% of missing nodes


In [33]:
import numpy as np
from collections import Counter
import ltlf_decode as LD

# enforce only rules that are (almost) always true on the training traces
cnt = Counter(tuple(s) for s in full_seqs)
train_sup = np.array([sum(c for s, c in cnt.items() if d.accepts(s)) / len(full_seqs) for d in dfas])
print("rules by train support:", np.round(np.sort(train_sup)[::-1][:60:6], 4))

keys = list(outputcat) + list(outputreal)
for SUP_MIN in (0.98, 0.99, 0.999):
    rule_idx = [i for i, s in enumerate(train_sup) if s >= SUP_MIN]
    decoder = LD.HardDecoder(dfas, A_REAL, rule_idx=rule_idx)
    r = LD.evaluate_hard(net, X_test, decoder, CLASS_IDS, MISSING_ID, N_CLASSES, keys, device,
                         mask_to_idx, DataLoader, beam=16)
    print(f"SUP_MIN={SUP_MIN} ({len(rule_idx)} rules):", {k: round(v, 4) for k, v in r.items()})


rules by train support: [1.     1.     1.     1.     1.     0.9986 0.9971 0.9855 0.9826]
SUP_MIN=0.98 (49 rules): {'acc_plain': 0.6746, 'acc_hard': 0.7426, 'sat_plain': 0.9898, 'sat_hard': 1.0, 'traces_changed': 0.2118, 'fallback': 0.0015, 'n_missing': 1985}
SUP_MIN=0.99 (39 rules): {'acc_plain': 0.6746, 'acc_hard': 0.7426, 'sat_plain': 0.9885, 'sat_hard': 1.0, 'traces_changed': 0.2088, 'fallback': 0.0015, 'n_missing': 1985}
SUP_MIN=0.999 (29 rules): {'acc_plain': 0.6746, 'acc_hard': 0.7259, 'sat_plain': 0.9943, 'sat_hard': 1.0, 'traces_changed': 0.1485, 'fallback': 0.0, 'n_missing': 1985}
